# 10 - WEO Excel Machine Learning Prediction Lab

This lab builds a practical predictive model from the WEO Excel workbook. The use case is: **predict next-year real GDP growth** from current-year macroeconomic indicators such as inflation, unemployment, debt, current account balance, savings, investment, trade growth, and GDP per capita.

## Important: Use a Python Notebook Runtime

Run this notebook with a **Python 3** kernel in Jupyter, VS Code, JupyterLab, or Google Colab. Do not run these cells in SQL Server query mode.

This notebook uses the WEO Excel workbook from Module 4. Locally, it looks for `Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx`. In Google Colab, upload `WEOApr2026all.xlsx` into the session files panel or use the upload prompt in the setup cell.

## 0. Setup

In [ ]:
%pip install pandas numpy scipy matplotlib seaborn scikit-learn openpyxl joblib -q

import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")


from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def find_weo_workbook():
    """Find the WEO workbook whether the notebook runs from the lab folder, repo root, or Colab."""
    candidate_paths = [
        Path("../../../Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("data/WEOApr2026all.xlsx"),
        Path("/content/WEOApr2026all.xlsx"),
        Path("WEOApr2026all.xlsx"),
    ]
    for path in candidate_paths:
        if path.exists():
            return path

    # Colab fallback: ask the learner to upload the Excel file if it is missing.
    try:
        from google.colab import files
        print("Upload WEOApr2026all.xlsx from Module 4 to continue.")
        uploaded = files.upload()
        if "WEOApr2026all.xlsx" in uploaded:
            return Path("WEOApr2026all.xlsx")
    except Exception:
        pass

    raise FileNotFoundError(
        "WEOApr2026all.xlsx was not found. Run locally from the repo, or upload the Excel file in Google Colab."
    )


WEO_PATH = find_weo_workbook()
print("Using workbook:", WEO_PATH)

excel_file = pd.ExcelFile(WEO_PATH)
print("Workbook sheets:", excel_file.sheet_names)

## 1. Build the Country-Year DataFrame

In [ ]:
INDICATOR_LABELS = {
    "NGDP_RPCH": "gdp_growth_pct",
    "PCPIPCH": "inflation_pct",
    "LUR": "unemployment_rate",
    "BCA_NGDPD": "current_account_pct_gdp",
    "GGXWDG_NGDP": "government_debt_pct_gdp",
    "NGDPDPC": "gdp_per_capita_usd",
    "NID_NGDP": "investment_pct_gdp",
    "NGSD_NGDP": "savings_pct_gdp",
    "TX_RPCH": "export_volume_growth_pct",
    "TM_RPCH": "import_volume_growth_pct",
}


def get_year_columns(frame):
    """Return Excel columns that represent years such as 1980, 2024, or 2031."""
    return [column for column in frame.columns if isinstance(column, int) or str(column).isdigit()]


def load_weo_sheets(path):
    """Load the useful WEO workbook sheets into separate DataFrames."""
    countries = pd.read_excel(path, sheet_name="Countries")
    country_groups = pd.read_excel(path, sheet_name="Country Groups")
    commodity_prices = pd.read_excel(path, sheet_name="Commodity Prices")
    group_composition = pd.read_excel(path, sheet_name="Country Group Composition")
    return countries, country_groups, commodity_prices, group_composition


def make_long_indicator_data(frame, indicator_ids, source_name):
    """Convert WEO wide year columns into a tidy row-per-country-year format."""
    year_columns = get_year_columns(frame)
    id_columns = ["COUNTRY.ID", "COUNTRY", "INDICATOR.ID", "INDICATOR", "UNIT"]
    filtered = frame.loc[frame["INDICATOR.ID"].isin(indicator_ids), id_columns + year_columns].copy()

    long_df = filtered.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name="year",
        value_name="value",
    )
    long_df["year"] = long_df["year"].astype(int)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df["source_sheet"] = source_name
    return long_df


def make_country_macro_dataframe(countries, group_composition):
    """Create one country-year row with selected WEO indicators as columns."""
    long_df = make_long_indicator_data(countries, list(INDICATOR_LABELS), "Countries")

    macro = (
        long_df.pivot_table(
            index=["COUNTRY.ID", "COUNTRY", "year"],
            columns="INDICATOR.ID",
            values="value",
            aggfunc="first",
        )
        .reset_index()
        .rename(columns=INDICATOR_LABELS)
    )

    advanced_codes = set(group_composition.loc[group_composition["groupname"].eq("Advanced Economies"), "countrycode"])
    emerging_codes = set(group_composition.loc[group_composition["groupname"].eq("Emerging Market and Developing Economies"), "countrycode"])
    ssa_codes = set(group_composition.loc[group_composition["groupname"].eq("Sub-Saharan Africa (SSA)"), "countrycode"])

    macro["economic_group"] = np.select(
        [macro["COUNTRY.ID"].isin(advanced_codes), macro["COUNTRY.ID"].isin(emerging_codes)],
        ["Advanced Economies", "Emerging Market and Developing Economies"],
        default="Other",
    )
    macro["is_sub_saharan_africa"] = macro["COUNTRY.ID"].isin(ssa_codes)
    return macro, long_df


countries_raw, country_groups_raw, commodity_prices_raw, group_composition = load_weo_sheets(WEO_PATH)
country_macro, country_long = make_country_macro_dataframe(countries_raw, group_composition)

print("Countries sheet rows:", len(countries_raw))
print("Country-year analytical rows:", len(country_macro))
display(country_macro.head())

## 2. Create the Prediction Dataset

For each country and year, the feature columns describe the current year. The target column is the next year's GDP growth. This is a simple supervised learning setup.

In [ ]:
model_df = country_macro.sort_values(["COUNTRY.ID", "year"]).copy()

# Target: next year's real GDP growth for the same country.
model_df["next_year_gdp_growth_pct"] = model_df.groupby("COUNTRY.ID")["gdp_growth_pct"].shift(-1)

feature_columns = [
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_rate",
    "government_debt_pct_gdp",
    "current_account_pct_gdp",
    "investment_pct_gdp",
    "savings_pct_gdp",
    "export_volume_growth_pct",
    "import_volume_growth_pct",
    "gdp_per_capita_usd",
]
target_column = "next_year_gdp_growth_pct"

# Use years with enough modern coverage. The target for 2031 is missing because 2032 is not in the workbook.
ml_df = model_df[model_df["year"].between(2010, 2030)].dropna(subset=[target_column]).copy()

print("ML rows:", len(ml_df))
print("Countries:", ml_df["COUNTRY"].nunique())
display(ml_df[["COUNTRY", "year"] + feature_columns + [target_column]].head())

## 3. Train/Test Split

For time-oriented economic data, train on earlier years and test on later years. This is closer to real prediction than randomly mixing all years.

In [ ]:
train_df = ml_df[ml_df["year"] <= 2023].copy()
test_df = ml_df[ml_df["year"] >= 2024].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]
X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Test years:", sorted(test_df["year"].unique()))

## 4. Baseline Model: Linear Regression

The preprocessing pipeline fills missing values with the median and scales numeric features before fitting the model.

In [ ]:
linear_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear regression MAE:", round(linear_mae, 3))
print("Linear regression RMSE:", round(linear_rmse, 3))
print("Linear regression R2:", round(linear_r2, 3))

## 5. Second Model: Random Forest Regression

A random forest can capture non-linear relationships. It still uses median imputation because some WEO indicators have missing coverage for some countries.

In [ ]:
forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(n_estimators=250, random_state=42, min_samples_leaf=3)),
])

forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)

forest_mae = mean_absolute_error(y_test, forest_predictions)
forest_rmse = np.sqrt(mean_squared_error(y_test, forest_predictions))
forest_r2 = r2_score(y_test, forest_predictions)

print("Random forest MAE:", round(forest_mae, 3))
print("Random forest RMSE:", round(forest_rmse, 3))
print("Random forest R2:", round(forest_r2, 3))

## 6. Compare Models

In [ ]:
metrics = pd.DataFrame([
    {"model": "linear_regression", "metric": "mae", "value": linear_mae},
    {"model": "linear_regression", "metric": "rmse", "value": linear_rmse},
    {"model": "linear_regression", "metric": "r2", "value": linear_r2},
    {"model": "random_forest", "metric": "mae", "value": forest_mae},
    {"model": "random_forest", "metric": "rmse", "value": forest_rmse},
    {"model": "random_forest", "metric": "r2", "value": forest_r2},
]).round(3)

display(metrics)

# Choose the model with the lowest MAE for export and later testing.
best_model_name = "linear_regression" if linear_mae <= forest_mae else "random_forest"
best_model = linear_model if best_model_name == "linear_regression" else forest_model
print("Best model by MAE:", best_model_name)

plt.figure(figsize=(7, 4))
sns.barplot(data=metrics[metrics["metric"].isin(["mae", "rmse"])], x="metric", y="value", hue="model")
plt.title("WEO GDP Growth Model Error Comparison")
plt.xlabel("Metric")
plt.ylabel("Error in percentage points")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_ml_model_error_comparison.png", dpi=150)
plt.show()

## 7. Review Predictions

Compare actual and predicted next-year GDP growth for the test period.

In [ ]:
prediction_review = test_df[["COUNTRY", "year", "economic_group", target_column]].copy()
prediction_review["linear_prediction"] = linear_predictions
prediction_review["forest_prediction"] = forest_predictions
prediction_review["forest_error"] = prediction_review["forest_prediction"] - prediction_review[target_column]

display(prediction_review.sort_values("forest_error", key=lambda s: s.abs(), ascending=False).head(15).round(2))

plt.figure(figsize=(6, 6))
sns.scatterplot(data=prediction_review, x=target_column, y="forest_prediction", hue="economic_group", alpha=0.75)
min_value = min(prediction_review[target_column].min(), prediction_review["forest_prediction"].min())
max_value = max(prediction_review[target_column].max(), prediction_review["forest_prediction"].max())
plt.plot([min_value, max_value], [min_value, max_value], color="black", linewidth=1)
plt.title("Actual vs Predicted Next-Year GDP Growth")
plt.xlabel("Actual next-year growth (%)")
plt.ylabel("Predicted next-year growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_ml_actual_vs_predicted.png", dpi=150)
plt.show()

## 8. Feature Importance

For the random forest, feature importance gives a simple ranking of which inputs the model used most.

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": forest_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)

display(feature_importance)

plt.figure(figsize=(8, 5))
sns.barplot(data=feature_importance, x="importance", y="feature")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_ml_feature_importance.png", dpi=150)
plt.show()

## 9. Export the Model and Test One Country

This example loads the saved model and predicts next-year GDP growth for Lesotho using the latest available feature year in the test period.

In [ ]:
model_path = OUTPUT_DIR / "weo_next_year_gdp_growth_model.joblib"
dump(best_model, model_path)

loaded_model = load(model_path)

lesotho_rows = model_df[model_df["COUNTRY"].eq("Lesotho, Kingdom of")].sort_values("year")
latest_lesotho = lesotho_rows.dropna(subset=["gdp_growth_pct"]).tail(1)

if latest_lesotho.empty:
    print("Lesotho was not available in the prepared model data.")
else:
    lesotho_features = latest_lesotho[feature_columns]
    lesotho_prediction = loaded_model.predict(lesotho_features)[0]
    display(latest_lesotho[["COUNTRY", "year"] + feature_columns])
    print("Best model by MAE:", best_model_name)
    print("Predicted next-year GDP growth for Lesotho:", round(lesotho_prediction, 2), "%")
    print("Saved model:", model_path)


## 10. Export Model Outputs

In [ ]:
metrics_path = OUTPUT_DIR / "weo_ml_model_metrics.csv"
predictions_path = OUTPUT_DIR / "weo_ml_prediction_review.csv"
importance_path = OUTPUT_DIR / "weo_ml_feature_importance.csv"

metrics.to_csv(metrics_path, index=False)
prediction_review.round(4).to_csv(predictions_path, index=False)
feature_importance.to_csv(importance_path, index=False)

print("Saved:", metrics_path)
print("Saved:", predictions_path)
print("Saved:", importance_path)